# C-1N: your first policy

Edit a policy, record its motion, and compare measurements with its replay.
The code path is **this notebook → `c1n/learning.py` → `c1n/simulation.py`**.
The helper owns recording and display. You own the policy, observation, reward,
action timing, and later RL implementation.

Run the setup cell below. It loads the model and resets it; it does not advance physics.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "spider" / "learning.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel in the repository or lab/notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from spider.learning import LearningSimulation

sim = LearningSimulation()
measured = sim.reset()
print("Number of actuator targets:", sim.model.nu)
from spider.learning import record_policy, plot_recordings

print("Physics timestep (s):", sim.model.opt.timestep)
print("Measurement fields:", ", ".join(measured.__dataclass_fields__))
print("Actuators:", [(i, sim.model.actuator(i).name) for i in range(sim.model.nu)])


## Edit here: `policy(observation)`

Your draft is preserved below. The next task is to make it accept `MeasuredState`
and return 18 finite joint-target offsets in radians, in the actuator order shown above.
Zero offset means the neutral target for that actuator.

1. Choose a named measurement from `observation`; it is not a sequence.
2. Define its desired value separately from the measurement.
3. Choose the actuator that responds. Make both comparison branches address that actuator.
4. Return the offsets. Run the check cell after your edit.

We will review this function before interpreting a rollout. A policy call chooses
targets; gravity, contact forces, and actuator torques determine the resulting motion.

In [ ]:
def policy(observation):
    offsets = np.zeros(18, dtype=float)
    offsets[0] = 1
    offsets[1] = -1

    measured = observation[0]
    desired = observation[1]

    if measured > desired:
        offsets[0] = -1
    elif measured < desired:
        offsets[1] = -1
    return offsets


In [ ]:
action = np.asarray(policy(measured), dtype=float)
assert action.shape == (sim.model.nu,)
assert np.isfinite(action).all()
print(action)

## Record and inspect

These are provisional **inspection** settings: 10 physics steps per action and
100 actions. Change them deliberately; the printed duration is not an RL episode rule.
Give the run a label with the parameter values you changed. Each run resets the robot.
Reset your policy's own state too if you later make it stateful.

The viewer replays the same action endpoints used by the plots. It does not run the
policy again. Events between recorded endpoints can be missed. Close the viewer when done.

In [ ]:
physics_steps = 10
action_count = 100
label = "draft: add your changed parameter values"
print("Action interval (s):", physics_steps * sim.model.opt.timestep)
print("Inspection duration (s):", action_count * physics_steps * sim.model.opt.timestep)

recording = record_policy(policy, label=label,
                          physics_steps=physics_steps, action_count=action_count)
viewer = recording.watch(REPO_ROOT / "telemetry" / "policy-replays", speed=1.0)
fig, axes = plot_recordings(recording, actuator=0)

## Compare one change

Keep the old recording in another variable before recording a change.
Use `plot_recordings(control, recording, actuator=0)` to compare two runs.
An all-zero policy is the neutral-target control; the STAND controller is a separate treatment.
Watch both before interpreting the plots.

`recording.offsets` holds requested offsets. `recording.targets` holds clipped absolute
targets. `action_times` marks each hold's start; `measurements` includes reset and each
hold's end. The plots show displacement, height, contacts, and joint response; they do not
define a reward or establish learned walking.

Record your prediction, observed difference, and next change here.